# Train A2C Agent on Ms. Pac-Man

This notebook demonstrates training an Advantage Actor-Critic (A2C) agent.

In [ ]:
# For Google Colab
# !git clone https://github.com/SABRYOLA/pacman.git
# %cd pacman
# !pip install -r requirements.txt

In [ ]:
import torch
import yaml
import sys
sys.path.append('..')

from src.agents import A2CAgent
from src.environment import create_vec_env, create_env
from src.training import OnPolicyTrainer
from src.utils import Logger, CheckpointManager
from src.networks import get_device

In [ ]:
with open('../configs/a2c_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['total_timesteps'] = 100000
config['n_envs'] = 4

print('A2C Configuration:')
for k, v in config.items():
    print(f'  {k}: {v}')

In [ ]:
device = get_device(config['device'])
env = create_vec_env(config['env_name'], n_envs=config['n_envs'], seed=config['seed'])
eval_env = create_env(config['env_name'], seed=config['seed'] + 1000)
n_actions = env.single_action_space.n

In [ ]:
agent = A2CAgent(
    n_actions=n_actions,
    learning_rate=config['learning_rate'],
    gamma=config['gamma'],
    n_steps=config['n_steps'],
    n_envs=config['n_envs'],
    ent_coef=config['ent_coef'],
    vf_coef=config['vf_coef'],
    device=device
)

In [ ]:
logger = Logger('../logs/a2c_notebook', 'A2C')
checkpoint_manager = CheckpointManager('../models', 'A2C')
trainer = OnPolicyTrainer(agent, env, logger, checkpoint_manager, config['total_timesteps'], eval_env=eval_env)

In [ ]:
trainer.train()

In [ ]:
print(logger.get_stats())

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs/

In [ ]:
agent.save('../models/a2c_notebook_final.pt')
env.close()
eval_env.close()
logger.close()